In [ ]:
# Paths come from the memtune package (see docs/INSTALL.md): no machine-specific locations.
#   data   -> data/lbfgs_runs.csv   (override: MEMTUNE_DATA_DIR)
#   models -> models/              (override: MEMTUNE_MODELS_DIR)
#   output -> results/             (override: MEMTUNE_RESULTS_DIR)
from memtune import paths
(paths.results_dir() / "retrained_models").mkdir(parents=True, exist_ok=True)  # notebooks never overwrite models/


In [ ]:
import pandas as pd
import numpy as np
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 20)
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from scipy.stats import randint, uniform
import joblib
from sklearn.model_selection import cross_val_predict
import csv

### Data Preprocessing

In [ ]:
df = pd.read_csv(str(paths.dataset_path()))
first = ["name", "nvar"]
df = df[first + [c for c in df.columns if c not in first]]
df["nvar"].value_counts
df = df.sort_values(["name", "nvar", "mem"]).reset_index()
df

In [ ]:
feature_cols_reg = [
                    # Core Features
                    "nvar", 
                    "mem",
                    "tree_length", 
                    "tree_depth", 
                    "time_obj",  # initial eval
                    "time_grad",
                    "extract_duration_problem",
                    
                    # Expression Tree Features
                    "highest_degree (ExprTree)",
                    "count_plus_minus",
                    "count_trigonometric_function",
                    "count_exponential_function",
                    "count_op_function",

                    # allocation
                    "alloc_obj",
                    "alloc_grad",
                    "total_alloc",
                  ]

target_cols_reg = [ "neval_obj",
                    "neval_grad",
                    "timed_bytes"] #(neval_obj, neval_grad, timed_bytes)

target_col_model = ["stats_elapsed_time"]

group_key = ["name", "nvar"]                                                                                                                   # [...]     
all_cols = list(set(feature_cols_reg + target_cols_reg + target_col_model + group_key))
solver_metrics = ['neval_grad', 'neval_obj', 'timed_bytes']
df[solver_metrics] = df[solver_metrics].fillna(0)
df = df[all_cols]
df.isna().sum().sort_values(ascending=False)

In [ ]:
df = df.copy()
df["mem_count"] = df.groupby(["name", "nvar"]).transform('size')
df = df[df["mem_count"] == 100].copy().reset_index(drop=True)
df

In [ ]:
# unique instances: one row per (problem, nvar)
instances = df[group_key].drop_duplicates()

# shuffle instances
instances = instances.sample(frac=1, random_state=66).reset_index(drop=True)

n = len(instances)
n_train = int(0.8 * n)

train_inst = instances.iloc[:n_train]
test_inst  = instances.iloc[n_train :]

# assign rows to splits by (problem, nvar)
train_df = df.merge(train_inst, on=["nvar", "name"], how="inner").reset_index(drop=True)
test_df  = df.merge(test_inst,  on=["nvar", "name"], how="inner").reset_index(drop=True)
all_df = pd.concat([train_df, test_df], axis=0)
all_df = all_df.reset_index(drop=True)
all_df

In [ ]:
 # confirm and use the three predicted feature to predict time                                                                                                              #  
X_train = train_df[feature_cols_reg].to_numpy(dtype=float)
X_test  = test_df[feature_cols_reg].to_numpy(dtype=float)

y_train = np.log1p(train_df[target_cols_reg].to_numpy(dtype=float))
y_test  = np.log1p(test_df[target_cols_reg].to_numpy(dtype=float))

X_all = np.concatenate([X_train, X_test], axis=0)


### Load Stage 1 RF and GB and form the Best Stage 1 Hybrid Model

In [ ]:
rf_stage1 = joblib.load(str(paths.models_dir() / "stage1/random_forest.pkl"))
gb_stage1 = joblib.load(str(paths.models_dir() / "stage1/xgboost.pkl"))

In [ ]:
oof_train_rf = cross_val_predict(rf_stage1, X_train, y_train, cv=5, n_jobs=-1)
oof_train_gb = cross_val_predict(gb_stage1, X_train, y_train, cv=5, n_jobs=-1)

# Test set predictions are generated normally using the fully-fitted models
test_pred_rf = rf_stage1.predict(X_test)
test_pred_gb = gb_stage1.predict(X_test)

# Stack features in pure LOG-SPACE
X_train_time = np.column_stack([
    oof_train_rf[:, 0], 
    oof_train_rf[:, 1], 
    oof_train_gb[:, 2]
])

X_test_time = np.column_stack([
    test_pred_rf[:, 0], 
    test_pred_rf[:, 1], 
    test_pred_gb[:, 2]
])

In [ ]:
y_train_time = np.log1p(train_df[target_col_model].to_numpy(dtype=float)).ravel()
y_test_time = np.log1p(test_df[target_col_model].to_numpy(dtype=float)).ravel()

### Performance Ratio Global Function

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import gmean

def evaluate_model(model, label="Model"):
    """
    Evaluates a stage-2 model using global X_test_time and global test_df.
    Includes arithmetic mean, geometric mean, and maximum performance ratio metrics.
    """
    predictions = model.predict(X_test_time)
    working_df = test_df.copy() 
    
    working_df["predicted_time"] = predictions
    group_cols = ["name", "nvar"]

    # 1. Model Selection
    model_choices = (
        working_df.sort_values(by=group_cols + ["predicted_time", "mem"], ascending=[True, True, True, False])
        .groupby(group_cols).first().reset_index()
        [group_cols + ["mem", "predicted_time", "stats_elapsed_time"]]
        .rename(columns={"mem": "predicted_mem", "stats_elapsed_time": "actual_time"})
    )

    # 2. Oracle Selection
    oracle_choices = (
        working_df.sort_values(by=group_cols + ["stats_elapsed_time", "mem"], ascending=[True, True, True, False])
        .groupby(group_cols).first().reset_index()
        [group_cols + ["mem", "stats_elapsed_time"]]
        .rename(columns={"mem": "best_mem", "stats_elapsed_time": "best_time"})
    )

    # 3. Merge Matrix & Apply Hardened 0.10s Absolute Tolerance Filter
    final_df = pd.merge(model_choices, oracle_choices, on=group_cols).copy()
    final_df['actual_time'] = pd.to_numeric(final_df['actual_time'], errors='coerce')
    final_df['best_time'] = pd.to_numeric(final_df['best_time'], errors='coerce')

    final_df['raw_ratio'] = final_df['actual_time'] / final_df['best_time']
    time_diff = np.abs(final_df['actual_time'] - final_df['best_time'])
    final_df['performance_ratio'] = np.where(time_diff <= 0.10, 1.0, final_df['raw_ratio'])

    # Reorder display columns
    column_order = group_cols + ["predicted_mem", "predicted_time", "actual_time", "best_mem", "best_time", "performance_ratio", "raw_ratio"]
    final_df = final_df[column_order]

    # Output Summary Blocks
    print("\n" + "="*145)
    print(f"PER-PROBLEM PERFORMANCE PROFILE SUMMARY ({label.upper()})")
    print("="*145)
    print(final_df.to_string(index=False, formatters={
        'predicted_time': '{:.4f}'.format, 'actual_time': '{:.4f}'.format,
        'best_time': '{:.4f}'.format, 'performance_ratio': '{:.4f}'.format, 'raw_ratio': '{:.4f}'.format
    }))
    print("="*145)

    total_actual = final_df['actual_time'].sum()
    total_best = final_df['best_time'].sum()
    
    # Calculate Core Evaluation Statistics
    global_mean_ratio = final_df['performance_ratio'].mean()
    geom_mean_ratio = gmean(final_df['performance_ratio'])  # Clean Scipy implementation
    max_performance_ratio = final_df['performance_ratio'].max()
    
    print("\n" + "="*50)
    print(f"AGGREGATE METRICS ({label.upper()})")
    print("="*50)
    print(final_df['performance_ratio'].quantile([0.50, 0.75, 0.90]).to_string(float_format='{:.4f}'.format))
    print("-"*50)
    print(f"Global Mean Perf. Ratio     : {global_mean_ratio:.4f}")
    print(f"Geometric Mean Perf. Ratio  : {geom_mean_ratio:.4f}")
    print(f"Maximum Performance Ratio   : {max_performance_ratio:.4f}")
    print("-"*50)
    print(f"Total Actual Compute Time   : {total_actual:.4f}s")
    print(f"Total Oracle Optimal Time   : {total_best:.4f}s")
    print("="*50)
    
    return final_df

## Train Model directly on Best Stage 1 Hybrid Model Predicted Features

### Using Predicted Data from Best Stage 1 Hybrid Model For Linear Regression

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict
import joblib  # Standard library for saving scikit-learn models

In [ ]:
# Initialize Stage 2 Linear Regression Regressor
lr_predictor = LinearRegression()

# 1. 5-Fold Cross-Validation Step (on Training Data in log-space)
cv_pred_log = cross_val_predict(lr_predictor, X_train_time, y_train_time, cv=5)
cv_mse = mean_squared_error(y_train_time, cv_pred_log)
cv_r2 = r2_score(y_train_time, cv_pred_log)

print("--- 5-Fold Cross-Validation Metrics (Train Set - Log Scale) ---")
print(f"CV MSE = {cv_mse:.4f} | CV R² = {cv_r2:.4f}")

# 2. Final Fit and Test Prediction
lr_predictor.fit(X_train_time, y_train_time)
pred_log = lr_predictor.predict(X_test_time)

# 3. Inverse-Transform to Original Physical Scale for Evaluation Metrics
# y_test_orig = np.expm1(y_test_time)
# pred_orig = np.expm1(pred_log)

test_mse = mean_squared_error(y_test_time, pred_log)
test_r2 = r2_score(y_test_time, pred_log)

print("\n--- Final Test Set Metrics (Original Physical Scale) ---")
print(f"MSE = {test_mse:.4f} | R² = {test_r2:.4f}")

# 4. Quantiled Relative Error Distribution (Original Scale)
relative_error = np.abs(pred_log - y_test_time) / (np.abs(y_test_time) + 1e-8)

quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
print(f"{target_col_model}:")
print(f"Quantile (q) | Relative Error Value Q(q)")

q_values = np.quantile(relative_error, quantiles)
for q, val in zip(quantiles, q_values):
    print(f"  Q({q:.2f}):  {val:.4f}")

In [ ]:
evaluate_model(lr_predictor)

### Using Predicted Data from Best Stage 1 Hybrid Model For Random Forest

In [ ]:
# import numpy as np
# import pandas as pd
# from scipy.stats import randint, uniform
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import RandomizedSearchCV

# # ==============================================================================
# # SEARCH SPACE CONFIGURATION
# # ==============================================================================
# param_distributions = {
#     'n_estimators': randint(100, 300),
#     'max_depth': [None, 10, 20, 30],
#     'min_samples_split': randint(2, 10),
#     'min_samples_leaf': randint(1, 5),
#     'max_features': [0.5, 0.7, 'sqrt'],
#     'bootstrap': [True],
#     'min_impurity_decrease': uniform(0, 0.001), 
#     'ccp_alpha': uniform(0, 0.001),             
#     'criterion': ['squared_error', 'friedman_mse']
# }

# rf_stage2 = RandomForestRegressor(random_state=42, n_jobs=1)

# # Set n_iter=1000 and verbose=3 to print real-time fold updates
# time_random_search = RandomizedSearchCV(
#     rf_stage2,
#     param_distributions=param_distributions,
#     n_iter=1000,
#     cv=5,
#     scoring='neg_mean_squared_error',
#     n_jobs=4,
#     random_state=42,
#     verbose=3,
#     return_train_score=True,
#     error_score='raise'
# )

# print("Starting 1,000-iteration Randomized Search Cross-Validation...")
# print(f"Total Fits to Execute: {1000 * 5} (1000 iterations x 5 folds)\n")

# # Fit Stage 2 on log features to predict your log targets
# time_random_search.fit(X_train_time, y_train_time)

# # ==============================================================================
# # PROGRESSION LANDSCAPE ANALYSIS (Print Data Along the Way)
# # ==============================================================================
# results_df = pd.DataFrame(time_random_search.cv_results_)
# results_df['actual_mse'] = -results_df['mean_test_score']

# print("\n" + "="*80)
# print("RANDOM SEARCH SEARCH-SPACE LANDSCAPE (MILESTONE INTERVALS)")
# print("="*80)
# print("Checking performance distribution across the 1,000 runs:")

# # Print structural metrics at regular intervals across the search trajectory
# intervals = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 999]
# milestone_cols = ['actual_mse', 'mean_fit_time', 'param_n_estimators', 
#                   'param_max_depth', 'param_min_samples_split', 'param_min_samples_leaf']

# sorted_by_run = results_df.copy() # Keep sequential run order intact for tracking
# print(sorted_by_run[milestone_cols].iloc[intervals].to_string(index=True))

# print("\n" + "="*80)
# print("BEST HYPERPARAMETERS FOUND")
# print("="*80)
# print(f"Best Parameters : {time_random_search.best_params_}")
# print(f"Best Log-MSE    : {-time_random_search.best_score_:.4f}")
# print("="*80)

# # Export full log profile for deep investigation
# results_df.to_csv('rf_stage2_1000runs_trajectory.csv', index=False)

In [ ]:
time_best_model_rf = joblib.load(str(paths.models_dir() / "stage2_hybrid/random_forest.pkl"))
# time_best_model_rf = time_random_search.best_estimator_
accuracy = time_best_model_rf.score(X_test_time, y_test_time)
print(f"Time prediction accuracy based on predicted values={accuracy}")
# joblib.dump(time_best_model_rf, str(paths.results_dir() / "retrained_models" / "best_hybrid_rf_model_time_new_cv.pkl"))
# joblib.dump(time_best_model_rf, str(paths.results_dir() / "retrained_models" / "best_hybrid_rf_model_time.pkl"))

In [ ]:
pred = time_best_model_rf.predict(X_test_time)
test_mse = mean_squared_error(y_test_time, pred)
test_r2  = r2_score(y_test_time, pred)
print(f"MSE={test_mse:.4f}  R²={test_r2:.4f}")

# Compute pointwise relative error matching the paper's formula: |y - y_hat| / (|y| + delta)
relative_error = np.abs(pred - y_test_time) / (np.abs(y_test_time) + 1e-8)

# Quantiled Relative Error Distribution
quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
for i, col in enumerate(target_col_model):
    print(f"\n{col}:")
    print(f"Quantile (q) | Relative Error Value Q(q)")
    
    # Safe extraction whether relative_error is a 1D vector or a 2D column matrix
    error_data = relative_error[:, i] if relative_error.ndim > 1 else relative_error
    q_values = np.quantile(error_data, quantiles)
    
    for q, val in zip(quantiles, q_values):
        print(f"  Q({q:.2f}):  {val:.4f}")
print(list(pred))

In [ ]:
evaluate_model(joblib.load(str(paths.models_dir() / "stage2_hybrid/random_forest.pkl")))
# evaluate_model(joblib.load(str(paths.models_dir() / "legacy/best_hybrid_rf_model_time.pkl")))

### Using Predicted Data from Best Stage 1 Hybrid Model For XGBoost

In [ ]:
# import numpy as np
# import pandas as pd
# import xgboost as xgb
# from scipy.stats import randint, uniform
# from sklearn.model_selection import RandomizedSearchCV

# # ==============================================================================
# # ROBUST SEARCH SPACE FOR LOG-SPACE META-MODELING
# # ==============================================================================
# param_distributions = {
#     'n_estimators':      randint(100, 500),
#     'learning_rate':     uniform(0.01, 0.15),  # Lower, stable rates to prevent variance spikes
#     'max_depth':         randint(2, 6),        # Shallow trees prevent overfitting to Stage 1 noise
#     'subsample':         uniform(0.6, 0.4),    # Stochastic sampling for generalization
#     'colsample_bytree':  uniform(0.6, 0.4),    
#     'min_child_weight':  randint(1, 10),
    
#     # Tightened structural constraints to match low-variance log targets
#     'gamma':             uniform(0, 0.1),      
#     'reg_alpha':         uniform(0, 0.1),      # L1 Regularization
#     'reg_lambda':        uniform(0.5, 1.5),    # L2 Regularization
# }

# # Utilizing hist tree method for extreme speed optimization across 5,000 total fits
# xgb_model = xgb.XGBRegressor(tree_method='hist', random_state=42, n_jobs=1)

# time_random_search = RandomizedSearchCV(
#     xgb_model,
#     param_distributions=param_distributions,
#     n_iter=1000, # 1,000 iterations across the CV loop
#     cv=5,
#     scoring='neg_mean_squared_error',
#     n_jobs=4,
#     random_state=42,
#     verbose=3,
#     return_train_score=True,
#     error_score='raise'
# )

# print("Starting 1,000-iteration Robust XGBoost Stage 2 Search...")
# print(f"Total Fits to Execute: {1000 * 5} (1000 iterations x 5 folds)\n")

# # Fit the robust model on your un-leaked out-of-fold log feature matrix
# time_random_search.fit(X_train_time, y_train_time)

# # ==============================================================================
# # PROGRESSION TRAJECTORY ANALYSIS (Print Data Along the Way)
# # ==============================================================================
# results_df = pd.DataFrame(time_random_search.cv_results_)
# results_df['actual_mse'] = -results_df['mean_test_score']

# print("\n" + "="*90)
# print("XGBOOST SEARCH-SPACE TRAJECTORY LANDSCAPE (MILESTONE INTERVALS)")
# print("="*90)
# print("Tracking hyperparameter convergence across sequential intervals:")

# # Extract performance indicators at regular milestones across the 1000 runs
# intervals = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 999]
# milestone_cols = ['actual_mse', 'mean_fit_time', 'param_n_estimators', 
#                   'param_learning_rate', 'param_max_depth', 'param_subsample',
#                   'param_reg_alpha', 'param_reg_lambda']

# sorted_by_run = results_df.copy() # Keeps the initial processing timeline visible
# print(sorted_by_run[milestone_cols].iloc[intervals].to_string(index=True))

# print("\n" + "="*80)
# print("OPTIMAL CONFIGURATION METRICS")
# print("="*80)
# print(f"Best Parameters : {time_random_search.best_params_}")
# print(f"Best Log-MSE    : {-time_random_search.best_score_:.4f}")
# print("="*80)

# # Export complete trajectory framework for performance logging
# results_df.to_csv('xgb_randomsearch_stage2_1000runs_trajectory.csv', index=False)

In [ ]:
time_best_model_gb = joblib.load(str(paths.models_dir() / "stage2_hybrid/xgboost.pkl"))
# time_best_model_gb = time_random_search.best_estimator_
accuracy = time_best_model_gb.score(X_test_time, y_test_time)
print(f"Time prediction accuracy based on predicted values={accuracy}")

# joblib.dump(time_best_model_gb, str(paths.results_dir() / "retrained_models" / "best_hybrid_xgb_model_time_new_cv.pkl"))

In [ ]:
pred = time_best_model_gb.predict(X_test_time)
test_mse = mean_squared_error(y_test_time, pred)
test_r2  = r2_score(y_test_time, pred)
print(f"MSE={test_mse:.4f}  R²={test_r2:.4f}")

# Compute pointwise relative error matching the paper's formula: |y - y_hat| / (|y| + delta)
relative_error = np.abs(pred - y_test_time) / (np.abs(y_test_time) + 1e-8)

# Quantiled Relative Error Distribution
quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
for i, col in enumerate(target_col_model):
    print(f"\n{col}:")
    print(f"Quantile (q) | Relative Error Value Q(q)")
    
    # Safe extraction whether relative_error is a 1D vector or a 2D column matrix
    error_data = relative_error[:, i] if relative_error.ndim > 1 else relative_error
    q_values = np.quantile(error_data, quantiles)
    
    for q, val in zip(quantiles, q_values):
        print(f"  Q({q:.2f}):  {val:.4f}")

In [ ]:
evaluate_model(joblib.load(str(paths.models_dir() / "stage2_hybrid/xgboost.pkl")))

## Use Pre-trained Stage 2 Models From Our Empirical Dataset directly to predict time from Best Stage 1 Hybrid Predicted Features

In [ ]:
y_train_time = np.log1p(train_df[target_col_model].to_numpy(dtype=float)).ravel()
y_test_time = np.log1p(test_df[target_col_model].to_numpy(dtype=float)).ravel()

### Try Pre-Trained Stage 2 Linear Regression Model

In [ ]:
# Load the saved model
lr_stage2 = joblib.load(str(paths.models_dir() / "stage2_ground_truth/linear_regression.pkl"))

# Use the LOADED model instance to predict
pred_log = lr_stage2.predict(X_test_time)

test_mse = mean_squared_error(y_test_time, pred_log)
test_r2 = r2_score(y_test_time, pred_log)

print("\n--- Final Test Set Metrics (Original Physical Scale) ---")
print(f"MSE = {test_mse:.4f} | R² = {test_r2:.4f}")

# 4. Quantiled Relative Error Distribution (Original Scale)
relative_error = np.abs(pred_log - y_test_time) / (np.abs(y_test_time) + 1e-8)

quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
print(f"{target_col_model}:")
print(f"Quantile (q) | Relative Error Value Q(q)")

q_values = np.quantile(relative_error, quantiles)
for q, val in zip(quantiles, q_values):
    print(f"  Q({q:.2f}):  {val:.4f}")

In [ ]:
evaluate_model(joblib.load(str(paths.models_dir() / "stage2_ground_truth/linear_regression.pkl")))

### Try Pre-Trained Stage 2 Random Forest

In [ ]:
# Load the saved model
rf_stage2 = joblib.load(str(paths.models_dir() / "stage2_ground_truth/random_forest.pkl"))

# Use the LOADED model instance to predict
pred_log = rf_stage2.predict(X_test_time)

test_mse = mean_squared_error(y_test_time, pred_log)
test_r2 = r2_score(y_test_time, pred_log)

print("\n--- Final Test Set Metrics (Original Physical Scale) ---")
print(f"MSE = {test_mse:.4f} | R² = {test_r2:.4f}")

# 4. Quantiled Relative Error Distribution (Original Scale)
relative_error = np.abs(pred_log - y_test_time) / (np.abs(y_test_time) + 1e-8)

quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
print(f"{target_col_model}:")
print(f"Quantile (q) | Relative Error Value Q(q)")

q_values = np.quantile(relative_error, quantiles)
for q, val in zip(quantiles, q_values):
    print(f"  Q({q:.2f}):  {val:.4f}")

In [ ]:
evaluate_model(joblib.load(str(paths.models_dir() / "stage2_ground_truth/random_forest.pkl")))

### Try Pre-Trained Stage 2 Gradient Boosting

In [ ]:
# Load the saved model
gb_stage2 = joblib.load(str(paths.models_dir() / "stage2_ground_truth/xgboost.pkl"))

# Use the LOADED model instance to predict
pred_log = gb_stage2.predict(X_test_time)

test_mse = mean_squared_error(y_test_time, pred_log)
test_r2 = r2_score(y_test_time, pred_log)

print("\n--- Final Test Set Metrics (Original Physical Scale) ---")
print(f"MSE = {test_mse:.4f} | R² = {test_r2:.4f}")

# 4. Quantiled Relative Error Distribution (Original Scale)
relative_error = np.abs(pred_log - y_test_time) / (np.abs(y_test_time) + 1e-8)

quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
print(f"{target_col_model}:")
print(f"Quantile (q) | Relative Error Value Q(q)")

q_values = np.quantile(relative_error, quantiles)
for q, val in zip(quantiles, q_values):
    print(f"  Q({q:.2f}):  {val:.4f}")

In [ ]:
evaluate_model(joblib.load(str(paths.models_dir() / "stage2_ground_truth/xgboost.pkl")))

In [ ]:
import matplotlib.pyplot as plt
xgb_importances_time = joblib.load(str(paths.models_dir() / "stage2_ground_truth/xgboost.pkl")).feature_importances_
indices_xgb = np.argsort(xgb_importances_time)[::-1]

# target_cols_reg = ["neval_obj", "neval_grad", "memory_usage"]

plt.figure(figsize=(10, 5))
plt.bar(range(len(target_cols_reg)), xgb_importances_time[indices_xgb])
plt.xticks(range(len(target_cols_reg)), [target_cols_reg[i] for i in indices_xgb], rotation=45, ha="right")
plt.ylabel("Feature importance")
plt.title("XGBoost feature importances (Stage 2)")
plt.tight_layout()
plt.show()

## Baseline Performance

In [ ]:
import pandas as pd

print("--------------------------------------------------------------------\n"
      "Performance For Static Heuristic Baselines (Mem=5 vs. Mem=87)")

group_cols = ["name", "nvar"]

# 1. Oracle Selection (Heuristic: Min actual time, break ties with LARGEST mem)
oracle_choices = (
    test_df.sort_values(by=group_cols + ["stats_elapsed_time", "mem"], ascending=[True, True, True, False])
    .groupby(group_cols).first().reset_index()
    [group_cols + ["mem", "stats_elapsed_time"]]
    .rename(columns={"mem": "best_mem", "stats_elapsed_time": "best_time"})
)

# 2. Extract actual recorded times when mem = 5 (Baseline 1: JSOSolver default)
baseline_5 = (
    test_df[test_df["mem"] == 5][group_cols + ["stats_elapsed_time"]]
    .rename(columns={"stats_elapsed_time": "time_mem5"})
)

# 3. Extract actual recorded times when mem = 87 (Baseline 2: Global Best)
baseline_87 = (
    test_df[test_df["mem"] == 87][group_cols + ["stats_elapsed_time"]]
    .rename(columns={"stats_elapsed_time": "time_mem87"})
)

# 4. Merge baseline profiles into a unified comparison matrix
comparison_df = pd.merge(oracle_choices, baseline_5, on=group_cols, how="left")
comparison_df = pd.merge(comparison_df, baseline_87, on=group_cols, how="left")

# 5. Calculate individual performance ratios (Actual Baseline Time / True Oracle Time)
comparison_df["ratio_mem5"] = comparison_df["time_mem5"] / comparison_df["best_time"]
comparison_df["ratio_mem87"] = comparison_df["time_mem87"] / comparison_df["best_time"]

# Reorder columns for scannability
print_cols = group_cols + ["best_mem", "best_time", "time_mem5", "ratio_mem5", "time_mem87", "ratio_mem87"]
final_baseline_df = comparison_df[print_cols]

# ==============================================================================
# VISUALIZATION & OUTPUT
# ==============================================================================

# 1. Output granular per-problem table
print("\n" + "="*145)
print("PER-PROBLEM STATIC BASELINE SIDE-BY-SIDE COMPARISON")
print("="*145)
print(final_baseline_df.to_string(index=False, formatters={
    'best_time': '{:.4f}'.format,
    'time_mem5': '{:.4f}'.format,
    'ratio_mem5': '{:.4f}'.format,
    'time_mem87': '{:.4f}'.format,
    'ratio_mem87': '{:.4f}'.format
}))
print("="*145)

# 2. Compute aggregate metrics and Global Performance Ratios
quantiles = [0.25, 0.50, 0.75, 0.90]
summary_df = pd.DataFrame({
    "Quantile (q)": quantiles,
    "Baseline 1 (mem=5)": final_baseline_df["ratio_mem5"].quantile(quantiles).values,
    "Baseline 2 (mem=87)": final_baseline_df["ratio_mem87"].quantile(quantiles).values
})

total_best_time = final_baseline_df["best_time"].sum()
total_time_mem5 = final_baseline_df["time_mem5"].sum()
total_time_mem87 = final_baseline_df["time_mem87"].sum()

global_ratio_mem5 = total_time_mem5 / total_best_time
global_ratio_mem87 = total_time_mem87 / total_best_time

print("\n" + "="*70)
print("AGGREGATE BASELINE PERFORMANCE STATISTICS")
print("="*70)
print(summary_df.to_string(index=False, formatters={
    'Baseline 1 (mem=5)': '{:.4f}'.format,
    'Baseline 2 (mem=87)': '{:.4f}'.format
}))
print("-"*70)
print(f"Total Compute Time (mem=5)        : {total_time_mem5:.4f}s")
print(f"Total Compute Time (mem=87)       : {total_time_mem87:.4f}s")
print(f"Total Oracle Optimal Time         : {total_best_time:.4f}s")
print("-"*70)
print(f"GLOBAL PERFORMANCE RATIO (mem=5)  : {global_ratio_mem5:.4f}")
print(f"GLOBAL PERFORMANCE RATIO (mem=87) : {global_ratio_mem87:.4f}")
print("="*70)

In [ ]:
class StaticBaselineModel:
    def __init__(self, target_mem):
        self.target_mem = target_mem
        
    def predict(self, X):
        # Force the selection logic to always pick the target memory tier
        return np.where(test_df["mem"] == self.target_mem, 0.0, 9999.0)

# ==============================================================================
# UNIFIED BASELINE EVALUATION RUNS
# ==============================================================================
evaluate_model(StaticBaselineModel(5), label="Baseline 1 (Mem=5)")
evaluate_model(StaticBaselineModel(87), label="Baseline 2 (Mem=87)")